<a href="https://colab.research.google.com/github/Bl00df1s7/GLDRUBF-Sentry/blob/main/%D0%B7%D0%BE%D0%BB%D0%BE%D1%82%D0%BE_%D0%B22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# 01 — INSTALLATION & IMPORTS
# ============================================================

!pip install t-tech-investments --index-url https://opensource.tbank.ru/api/v4/projects/238/packages/pypi/simple --quiet

import pandas as pd
import numpy as np
import requests

from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo


from t_tech.invest import Client
from t_tech.invest import CandleInterval

print("✅ Imports OK")

✅ Imports OK


In [4]:
import os
# ============================================================
# 02 — T-INVEST API CLIENT
# ============================================================

TOKEN = os.environ.get("T_SANDAPI", "")

if not TOKEN:
    raise RuntimeError("❌ Secret T-Sandapi не найден")

client = Client(TOKEN)

print("✅ T-Invest Client создан")
print(f"   Sandbox Mode: {'Да' if 'sand' in str(client).lower() else 'Нет'}")

✅ T-Invest Client создан
   Sandbox Mode: Нет


In [5]:
# ============================================================
# 03 — GLDRUBF INSTRUMENT DISCOVERY
# ============================================================

TARGET_TICKER = "GLDRUBF"

with Client(TOKEN) as services:
    response = services.instruments.futures()
    futures = response.instruments

instrument = None

for x in futures:
    if x.ticker.upper() == TARGET_TICKER:
        instrument = x
        break

if instrument is None:
    raise RuntimeError(f"❌ Фьючерс {TARGET_TICKER} не найден")

UID = instrument.uid
FIGI = instrument.figi
CLASS_CODE = instrument.class_code
LOT = instrument.lot
MIN_PRICE_INCREMENT = instrument.min_price_increment

print("=== INSTRUMENT INFO ===")
print(f"Ticker:       {instrument.ticker}")
print(f"Name:         {instrument.name}")
print(f"UID:          {UID}")
print(f"FIGI:         {FIGI}")
print(f"Lot:          {LOT}")
print(f"Min tick:     {MIN_PRICE_INCREMENT}")

/tmp/ipykernel_1794/3228911994.py:8: DeprecatedWarning: futures is deprecated as of 1.0.0.
  response = services.instruments.futures()


=== INSTRUMENT INFO ===
Ticker:       GLDRUBF
Name:         GLDRUBF Золото (rub)
UID:          b347fe28-0d2a-45bf-b3bd-cda8a6ac64e6
FIGI:         FUTGLDRUBF00
Lot:          1
Min tick:     Quotation(units=0, nano=100000000)


In [6]:
# ============================================================
# 04 — STRATEGY CONFIGURATION (UPDATED)
# ============================================================

TIMEFRAME = "4H"

# ------------------------------------------------------------
# TREND FILTER
# ------------------------------------------------------------
EMA_LEN = 100  # Глобальный тренд (~16 дней на 4H)

# ------------------------------------------------------------
# ENTRY SIGNAL
# ------------------------------------------------------------
DONCHIAN_LEN = 20  # Локальный пробой

# ------------------------------------------------------------
# VOLATILITY
# ------------------------------------------------------------
ATR_LEN = 14

# ------------------------------------------------------------
# RISK MANAGEMENT (ATR-BASED)
# ------------------------------------------------------------
SL_ATR = 2.5      # Стоп-лосс в ATR
TP_ATR = 5.0      # Тейк-профит в ATR (R:R = 2:1)
BE_ATR = 2.0      # Триггер безубытка в ATR

# ------------------------------------------------------------
# PARABOLIC SAR
# ------------------------------------------------------------
SAR_START = 0.03
SAR_INC = 0.02
SAR_MAX = 0.20

print("=== STRATEGY CONFIG ===")
print(f"Timeframe:     {TIMEFRAME}")
print()
print("TREND FILTER")
print(f"EMA:           {EMA_LEN}")
print()
print("ENTRY")
print(f"Donchian:      {DONCHIAN_LEN}")
print()
print("VOLATILITY")
print(f"ATR:           {ATR_LEN}")
print()
print("RISK (ATR-BASED)")
print(f"SL:            {SL_ATR} ATR")
print(f"TP:            {TP_ATR} ATR (R:R = {TP_ATR/SL_ATR:.2f}:1)")
print(f"BE trigger:    {BE_ATR} ATR")
print()
print("SAR")
print(f"Start:         {SAR_START}")
print(f"Increment:     {SAR_INC}")
print(f"Maximum:       {SAR_MAX}")

=== STRATEGY CONFIG ===
Timeframe:     4H

TREND FILTER
EMA:           100

ENTRY
Donchian:      20

VOLATILITY
ATR:           14

RISK (ATR-BASED)
SL:            2.5 ATR
TP:            5.0 ATR (R:R = 2.00:1)
BE trigger:    2.0 ATR

SAR
Start:         0.03
Increment:     0.02
Maximum:       0.2


In [7]:
# ============================================================
# 05 — CANDLE LOADER
# ============================================================

def quotation_to_float(value):
    """Безопасное преобразование Quotation в float."""

    if value is None:
        return np.nan

    if isinstance(value, (int, float, np.number)):
        return float(value)

    if hasattr(value, "units") and hasattr(value, "nano"):
        return float(value.units) + float(value.nano) / 1_000_000_000

    if hasattr(value, "value"):
        return float(value.value)

    return float(value)


def candle_to_row(candle):
    return {
        "time": candle.time,
        "open": quotation_to_float(candle.open),
        "high": quotation_to_float(candle.high),
        "low": quotation_to_float(candle.low),
        "close": quotation_to_float(candle.close),
        "volume": candle.volume,
    }


def load_recent_candles(uid, candles_count=200):
    """Загружает последние N свечей с учетом 90-дневных чанков."""

    now_utc = datetime.now(timezone.utc)

    # 4H = 6 свечей в сутки, добавляем запас
    days = int(candles_count / 6) + 10
    start_date = now_utc - timedelta(days=days)

    rows = []
    current = start_date
    chunk = timedelta(days=90)

    while current < now_utc:
        chunk_end = min(current + chunk, now_utc)

        with Client(TOKEN) as services:
            response = services.market_data.get_candles(
                instrument_id=uid,
                from_=current,
                to=chunk_end,
                interval=CandleInterval.CANDLE_INTERVAL_4_HOUR,
            )

        rows.extend(candle_to_row(candle) for candle in response.candles)
        current = chunk_end

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.drop_duplicates("time").sort_values("time").reset_index(drop=True)

    return df.tail(candles_count).reset_index(drop=True)

print("✅ Candle loader готов")

✅ Candle loader готов


In [8]:
# ============================================================
# 06 — INDICATORS (UPDATED WITH EMA FILTER)
# ============================================================

def calculate_atr(df, length):
    """Расчет Average True Range."""

    high = df["high"]
    low = df["low"]
    close = df["close"]
    prev_close = close.shift(1)

    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)

    return tr.rolling(length).mean()


def prepare_indicators(df):
    """Расчет всех индикаторов с фильтром тренда."""

    data = df.copy().reset_index(drop=True)

    # ========================================================
    # ATR (Волатильность)
    # ========================================================
    data["atr"] = calculate_atr(data, ATR_LEN)

    # ========================================================
    # MACRO TREND FILTER (EMA 100)
    # ========================================================
    data["ema_100"] = data["close"].ewm(span=EMA_LEN, adjust=False).mean()

    # ========================================================
    # DONCHIAN CHANNEL (Только предыдущие свечи)
    # ========================================================
    data["donchian_upper"] = data["high"].rolling(DONCHIAN_LEN).max().shift(1)
    data["donchian_lower"] = data["low"].rolling(DONCHIAN_LEN).min().shift(1)

    # ========================================================
    # ENTRY SIGNALS (С ФИЛЬТРОМ ТРЕНДА)
    # ========================================================
    # Long: пробой верхнего Дончиана + цена выше EMA 100
    data["long_signal"] = (
        (data["close"] > data["donchian_upper"]) &
        (data["close"] > data["ema_100"])
    )

    # Short: пробой нижнего Дончиана + цена ниже EMA 100
    data["short_signal"] = (
        (data["close"] < data["donchian_lower"]) &
        (data["close"] < data["ema_100"])
    )

    return data

print("✅ Indicators готовы (включая EMA 100)")

✅ Indicators готовы (включая EMA 100)


In [9]:
# ============================================================
# 07 — PARABOLIC SAR CALCULATION
# ============================================================

def calculate_sar(df, start, inc, maximum):
    """Расчет Parabolic SAR."""

    high = df["high"].to_numpy()
    low = df["low"].to_numpy()
    close = df["close"].to_numpy()
    n = len(df)

    sar = np.full(n, np.nan)
    ep = np.full(n, np.nan)
    af = np.full(n, np.nan)
    trend = np.ones(n, dtype=int)

    if n == 0:
        return sar, trend

    sar[0] = close[0]
    ep[0] = close[0]
    af[0] = start
    trend[0] = 1

    for i in range(1, n):
        prev_sar = sar[i - 1]
        prev_ep = ep[i - 1]
        prev_af = af[i - 1]
        prev_trend = trend[i - 1]

        # UP TREND
        if prev_trend == 1:
            current_sar = prev_sar + prev_af * (prev_ep - prev_sar)

            if i >= 2:
                current_sar = min(current_sar, low[i - 1], low[i - 2])
            else:
                current_sar = min(current_sar, low[i - 1])

            # REVERSAL
            if low[i] < current_sar:
                trend[i] = -1
                sar[i] = prev_ep
                ep[i] = low[i]
                af[i] = start
            # CONTINUE UP
            else:
                trend[i] = 1
                sar[i] = current_sar

                if high[i] > prev_ep:
                    ep[i] = high[i]
                    af[i] = min(prev_af + inc, maximum)
                else:
                    ep[i] = prev_ep
                    af[i] = prev_af

        # DOWN TREND
        else:
            current_sar = prev_sar + prev_af * (prev_ep - prev_sar)

            if i >= 2:
                current_sar = max(current_sar, high[i - 1], high[i - 2])
            else:
                current_sar = max(current_sar, high[i - 1])

            # REVERSAL
            if high[i] > current_sar:
                trend[i] = 1
                sar[i] = prev_ep
                ep[i] = high[i]
                af[i] = start
            # CONTINUE DOWN
            else:
                trend[i] = -1
                sar[i] = current_sar

                if low[i] < prev_ep:
                    ep[i] = low[i]
                    af[i] = min(prev_af + inc, maximum)
                else:
                    ep[i] = prev_ep
                    af[i] = prev_af

    return sar, trend

print("✅ Parabolic SAR готов")

✅ Parabolic SAR готов


In [10]:
# ============================================================
# 08 — MARKET DATA LOADING
# ============================================================

df_raw = load_recent_candles(UID, candles_count=200)

if df_raw.empty:
    raise RuntimeError("❌ Не удалось получить свечи GLDRUBF")

df = prepare_indicators(df_raw)

df["sar"], df["sar_trend"] = calculate_sar(
    df, SAR_START, SAR_INC, SAR_MAX
)

# ============================================================
# CURRENT TIME
# ============================================================

now_utc = datetime.now(timezone.utc)
MSK = ZoneInfo("Europe/Moscow")

# ============================================================
# FIND LAST CLOSED 4H CANDLE
# ============================================================

CANDLE_DURATION = timedelta(hours=4)
df["candle_close_time"] = df["time"] + CANDLE_DURATION

closed_candidates = df[df["candle_close_time"] <= now_utc].copy()

if closed_candidates.empty:
    raise RuntimeError("❌ Не найдена ни одна закрытая 4H свеча")

closed = closed_candidates.iloc[-1]
closed_index = closed_candidates.index[-1]

if closed_index == 0:
    raise RuntimeError("❌ Недостаточно истории для предыдущей закрытой свечи")

previous_closed = df.loc[closed_index - 1]

# ============================================================
# CURRENT PRICE
# ============================================================

with Client(TOKEN) as services:
    response = services.market_data.get_last_prices(instrument_id=[UID])

if not response.last_prices:
    raise RuntimeError("❌ Не удалось получить текущую цену GLDRUBF")

current_price = quotation_to_float(response.last_prices[0].price)

# ============================================================
# DISPLAY
# ============================================================

print("=== MARKET DATA ===")
print(f"Current UTC:       {now_utc}")
print(f"Current MSK:       {now_utc.astimezone(MSK)}")
print()
print(f"Last closed 4H:    {closed['time'].astimezone(MSK)}")
print(f"Candle close UTC:  {closed['candle_close_time']}")
print(f"Candle close MSK:  {closed['candle_close_time'].astimezone(MSK)}")
print()
print(f"Open:              {closed['open']:.2f}")
print(f"High:              {closed['high']:.2f}")
print(f"Low:               {closed['low']:.2f}")
print(f"Close:             {closed['close']:.2f}")
print(f"Current price:     {current_price:.2f}")
print()
print(f"ATR:               {closed['atr']:.2f}")
print(f"EMA 100:           {closed['ema_100']:.2f}")
print(f"SAR:               {closed['sar']:.2f}")
print(f"SAR trend:         {'LONG' if closed['sar_trend'] == 1 else 'SHORT'}")

/tmp/ipykernel_1794/563837685.py:51: DeprecatedWarning: get_candles is deprecated as of 1.0.0.
  response = services.market_data.get_candles(
/tmp/ipykernel_1794/3572024368.py:48: DeprecatedWarning: get_last_prices is deprecated as of 1.0.0.
  response = services.market_data.get_last_prices(instrument_id=[UID])


=== MARKET DATA ===
Current UTC:       2026-08-13 20:31:52.528539+00:00
Current MSK:       2026-08-13 23:31:52.528539+03:00

Last closed 4H:    2026-08-13 19:00:00+03:00
Candle close UTC:  2026-08-13 20:00:00+00:00
Candle close MSK:  2026-08-13 23:00:00+03:00

Open:              11715.50
High:              11738.90
Low:               11675.00
Close:             11692.90
Current price:     11699.00

ATR:               76.69
EMA 100:           10988.12
SAR:               11576.69
SAR trend:         LONG


In [11]:
# ============================================================
# 09 — ACCOUNTS DISCOVERY
# ============================================================

with Client(TOKEN) as services:
    accounts_response = services.users.get_accounts()

accounts = accounts_response.accounts

if not accounts:
    raise RuntimeError("❌ Для этого токена не найдено ни одного счёта")

print("=== ACCOUNTS ===")
print(f"Всего счетов: {len(accounts)}")
print()

for i, account in enumerate(accounts, start=1):
    print(f"{i}. ID: {account.id} | Name: {account.name} | Type: {account.type} | Status: {account.status}")

/tmp/ipykernel_1794/411620085.py:6: DeprecatedWarning: get_accounts is deprecated as of 1.0.0.
  accounts_response = services.users.get_accounts()


=== ACCOUNTS ===
Всего счетов: 10

1. ID: 2009052016 | Name: Брокерский счёт | Type: 1 | Status: 2
2. ID: 2163652650 | Name: Тест секьюр  | Type: 1 | Status: 2
3. ID: 2040428803 | Name: ИИС | Type: 2 | Status: 2
4. ID: 2113327027 | Name: Road to 1kk | Type: 1 | Status: 2
5. ID: 2035914981 | Name: Инвесткопилка | Type: 3 | Status: 2
6. ID: 2093956148 | Name: Счет | Type: 1 | Status: 2
7. ID: 2146139894 | Name: Портфельное инвестирование 1 | Type: 1 | Status: 2
8. ID: 2218152205 | Name: Смарт-счет | Type: 7 | Status: 2
9. ID: 2226775427 | Name: Счет под ключ | Type: 4 | Status: 2
10. ID: 2023149547 | Name: Счет под ключ 1 | Type: 4 | Status: 2


In [12]:
# ============================================================
# 10 — POSITION DISCOVERY
# ============================================================

print("=== GLDRUBF POSITIONS ===")
print()

position_qty = 0.0
position_direction = "NONE"
POSITION_ACCOUNT_ID = None
POSITION_ACCOUNT_NAME = None
gldrubf_position = None

with Client(TOKEN) as services:
    for account in accounts:
        account_id = account.id

        try:
            positions_response = services.operations.get_positions(account_id=account_id)
        except Exception as e:
            print(f"⚠️ Не удалось получить позиции для {account_id}: {e}")
            continue

        futures_positions = positions_response.futures

        for position in futures_positions:
            if position.ticker.upper() != TARGET_TICKER:
                continue

            balance = float(position.balance)

            if balance != 0:
                gldrubf_position = position
                position_qty = balance
                POSITION_ACCOUNT_ID = account_id
                POSITION_ACCOUNT_NAME = account.name
                position_direction = "LONG" if balance > 0 else "SHORT"

                print(f"🎯 GLDRUBF POSITION FOUND")
                print(f"   Account:         {POSITION_ACCOUNT_ID}")
                print(f"   Account name:    {POSITION_ACCOUNT_NAME}")
                print(f"   Quantity:        {position_qty}")
                print(f"   Direction:       {position_direction}")
                break

        if gldrubf_position is not None:
            break

print()
print("=" * 60)
print("FINAL POSITION STATE")
print("=" * 60)

if position_direction == "NONE":
    print("⚪ FINAL: NO GLDRUBF POSITION")
else:
    print(f"🟢 FINAL: GLDRUBF {position_direction} × {position_qty}")

=== GLDRUBF POSITIONS ===



/tmp/ipykernel_1794/1556288471.py:19: DeprecatedWarning: get_positions is deprecated as of 1.0.0.
  positions_response = services.operations.get_positions(account_id=account_id)


🎯 GLDRUBF POSITION FOUND
   Account:         2163652650
   Account name:    Тест секьюр 
   Quantity:        3.0
   Direction:       LONG

FINAL POSITION STATE
🟢 FINAL: GLDRUBF LONG × 3.0


In [13]:
# ============================================================
# 11 — POSITION STATE (STATELESS)
# ============================================================

if gldrubf_position is None:
    position_direction = "NONE"
    position_qty = 0.0
    position_account_id = None
    position_account_name = None
    entry_price = np.nan
    entry_atr = np.nan

else:
    position_qty = float(gldrubf_position.balance)
    position_direction = "LONG" if position_qty > 0 else "SHORT"
    position_account_id = POSITION_ACCOUNT_ID
    position_account_name = POSITION_ACCOUNT_NAME

    # Получаем среднюю цену позиции через portfolio
    entry_price = np.nan

    with Client(TOKEN) as services:
        portfolio = services.operations.get_portfolio(account_id=position_account_id)

    for portfolio_position in portfolio.positions:
        if portfolio_position.figi == gldrubf_position.figi:
            entry_price = quotation_to_float(portfolio_position.average_position_price)
            break

    if np.isnan(entry_price):
        print("⚠️ Не удалось получить среднюю цену GLDRUBF")

    # ATR берём с последней закрытой 4H свечи
    entry_atr = float(closed["atr"])

print("=== POSITION STATE ===")

if position_direction == "NONE":
    print("Position: NONE")
else:
    print(f"Position:       {position_direction}")
    print(f"Account:        {position_account_name}")
    print(f"Account ID:     {position_account_id}")
    print(f"Quantity:       {position_qty}")
    print(f"Entry:          {entry_price:.2f}" if not np.isnan(entry_price) else "Entry:      N/A")
    print(f"Entry ATR:      {entry_atr:.2f}")

/tmp/ipykernel_1794/264984923.py:23: DeprecatedWarning: get_portfolio is deprecated as of 1.0.0.
  portfolio = services.operations.get_portfolio(account_id=position_account_id)


=== POSITION STATE ===
Position:       LONG
Account:        Тест секьюр 
Account ID:     2163652650
Quantity:       3.0
Entry:          11659.70
Entry ATR:      76.69


In [14]:
# ============================================================
# 12 — DYNAMIC LEVELS CALCULATION (ATR-BASED)
# ============================================================

sl_price = np.nan
tp_price = np.nan
be_trigger = np.nan
sar_price = float(closed["sar"])

if position_direction in ("LONG", "SHORT"):
    if np.isnan(entry_price):
        print("⚠️ Нет средней цены позиции — уровни не рассчитываем")
    else:
        entry_price = float(entry_price)
        entry_atr = float(entry_atr)

        if position_direction == "LONG":
            base_sl = entry_price - (entry_atr * SL_ATR)
            tp_price = entry_price + (entry_atr * TP_ATR)
            be_trigger = entry_price + (entry_atr * BE_ATR)
        else:  # SHORT
            base_sl = entry_price + (entry_atr * SL_ATR)
            tp_price = entry_price - (entry_atr * TP_ATR)
            be_trigger = entry_price - (entry_atr * BE_ATR)

        # Stateless Break-Even Logic
        if position_direction == "LONG" and current_price >= be_trigger:
            sl_price = entry_price
        elif position_direction == "SHORT" and current_price <= be_trigger:
            sl_price = entry_price
        else:
            sl_price = base_sl

print("=== LEVELS ===")

if position_direction == "NONE":
    print("Position: NONE")
elif np.isnan(entry_price):
    print(f"Position: {position_direction}")
    print("Entry:    N/A")
else:
    print(f"Account:       {position_account_name}")
    print(f"Direction:     {position_direction}")
    print(f"Quantity:      {position_qty}")
    print(f"Entry:         {entry_price:.2f}")
    print(f"SL:            {sl_price:.2f}")
    print(f"TP:            {tp_price:.2f}")
    print(f"BE trigger:    {be_trigger:.2f}")
    print(f"SAR:           {sar_price:.2f}")

=== LEVELS ===
Account:       Тест секьюр 
Direction:     LONG
Quantity:      3.0
Entry:         11659.70
SL:            11467.99
TP:            12043.13
BE trigger:    11813.07
SAR:           11576.69


In [15]:
# ============================================================
# 13 — SENTRY DECISION (STATE MACHINE)
# ============================================================

print("=" * 70)
print("GLDRUBF QUANT SENTRY")
print("=" * 70)
print()

print(f"Time:           {now_utc.astimezone(MSK)}")
print(f"Current price:  {current_price:.2f}")
print(f"Closed 4H:      {closed['time'].astimezone(MSK)}")
print(f"Closed price:   {closed['close']:.2f}")
print(f"EMA 100:        {closed['ema_100']:.2f}")
print()

# ============================================================
# REGIME: FLAT (NO POSITION)
# ============================================================

if position_direction == "NONE":
    print("📊 REGIME: FLAT (Поиск точки входа)")
    print()
    print("SIGNAL")

    if closed["long_signal"]:
        print("🟢 LONG ENTRY (Пробой + Тренд)")

        entry_reference = current_price
        signal_sl = entry_reference - closed["atr"] * SL_ATR
        signal_tp = entry_reference + closed["atr"] * TP_ATR
        signal_be = entry_reference + closed["atr"] * BE_ATR

        print(f"   Entry reference: {entry_reference:.2f}")
        print(f"   SL:              {signal_sl:.2f}")
        print(f"   TP:              {signal_tp:.2f}")
        print(f"   BE trigger:      {signal_be:.2f}")

    elif closed["short_signal"]:
        print("🔴 SHORT ENTRY (Пробой + Тренд)")

        entry_reference = current_price
        signal_sl = entry_reference + closed["atr"] * SL_ATR
        signal_tp = entry_reference - closed["atr"] * TP_ATR
        signal_be = entry_reference - closed["atr"] * BE_ATR

        print(f"   Entry reference: {entry_reference:.2f}")
        print(f"   SL:              {signal_sl:.2f}")
        print(f"   TP:              {signal_tp:.2f}")
        print(f"   BE trigger:      {signal_be:.2f}")

    else:
        print("⚪ NO ENTRY SIGNAL (Флэт или откат к EMA)")

# ============================================================
# REGIME: IN_POSITION
# ============================================================

else:
    print(f"📊 REGIME: IN_POSITION ({position_direction})")
    print()

    if np.isnan(entry_price):
        print("⚠️ Средняя цена позиции неизвестна.")
        print("⚠️ EXIT CHECK НЕ ВЫПОЛНЯЕМ.")
        print()
        print("ACTION")
        print("⚠️ NO ACTION — не удалось определить Entry")

    else:
        print("LEVELS")
        print(f"   SL:             {sl_price:.2f}")
        print(f"   TP:             {tp_price:.2f}")
        print(f"   BE trigger:     {be_trigger:.2f}")
        print(f"   SAR:            {sar_price:.2f}")
        print()

        # EXIT CONDITIONS
        if position_direction == "LONG":
            hit_sl = current_price <= sl_price
            hit_tp = current_price >= tp_price
            sar_exit = closed["sar_trend"] == -1  # SAR стал выше цены
        else:  # SHORT
            hit_sl = current_price >= sl_price
            hit_tp = current_price <= tp_price
            sar_exit = closed["sar_trend"] == 1  # SAR стал ниже цены

        # ACTION
        print("ACTION")

        if hit_sl:
            print(f"🔴 EXIT — SL @ {sl_price:.2f}")
        elif hit_tp:
            print(f"🟢 EXIT — TP @ {tp_price:.2f}")
        elif sar_exit:
            print("🟡 EXIT — SAR (Тренд сломлен)")
        elif (position_direction == "LONG" and current_price >= be_trigger) or \
             (position_direction == "SHORT" and current_price <= be_trigger):
            print("🟡 BE TRIGGER REACHED (Позиция в безубытке)")
        else:
            print("🟢 HOLD (Тренд актуален)")

print()
print("=" * 70)

GLDRUBF QUANT SENTRY

Time:           2026-08-13 23:31:52.528539+03:00
Current price:  11699.00
Closed 4H:      2026-08-13 19:00:00+03:00
Closed price:   11692.90
EMA 100:        10988.12

📊 REGIME: IN_POSITION (LONG)

LEVELS
   SL:             11467.99
   TP:             12043.13
   BE trigger:     11813.07
   SAR:            11576.69

ACTION
🟢 HOLD (Тренд актуален)



In [16]:
import os
# ============================================================
# 14 — TELEGRAM SETUP
# ============================================================

BOT_TOKEN = os.environ.get("BOT_TOKEN", "")

if not BOT_TOKEN:
    raise RuntimeError("❌ Secret BOT_TOKEN не найден")


def telegram_get_chat_id():
    """Получение chat_id из последних сообщений бота."""

    url = f"https://api.telegram.org/bot{BOT_TOKEN}/getUpdates"
    response = requests.get(url, timeout=10)
    response.raise_for_status()

    data = response.json()

    if not data.get("ok"):
        raise RuntimeError(f"❌ Telegram API error: {data}")

    updates = data.get("result", [])

    if not updates:
        raise RuntimeError("❌ Telegram не вернул сообщений. Отправь боту /start и запусти ячейку снова.")

    for update in reversed(updates):
        message = update.get("message")

        if message and message.get("chat"):
            return message["chat"]["id"]

    raise RuntimeError("❌ Не найден chat_id. Отправь боту /start.")


TELEGRAM_CHAT_ID = telegram_get_chat_id()

print(f"✅ Telegram chat_id получен: {TELEGRAM_CHAT_ID}")


def telegram_send(message):
    """Отправка сообщения в Telegram."""

    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"

    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "HTML",
    }

    response = requests.post(url, json=payload, timeout=10)
    response.raise_for_status()

    data = response.json()

    if not data.get("ok"):
        raise RuntimeError(f"❌ Telegram API error: {data}")

    return data


def fmt_price(value):
    """Форматирование цены для Telegram."""
    return f"{float(value):,.2f}".replace(",", " ")

print("✅ Telegram функции готовы")

✅ Telegram chat_id получен: 591958455
✅ Telegram функции готовы


In [17]:
# ============================================================
# 15 — TELEGRAM SEND FINAL STATUS
# ============================================================

updated_msk = now_utc.astimezone(MSK)
closed_msk = closed["time"].astimezone(MSK)

# ============================================================
# MARKET BLOCK
# ============================================================

market_block = (
    f"💰 <b>Цена:</b>        {fmt_price(current_price)}\n"
    f"📊 <b>Закрытие 4H:</b> {fmt_price(closed['close'])}\n"
    f"📈 <b>EMA 100:</b>     {fmt_price(closed['ema_100'])}\n"
    f"⏱ <b>Свеча:</b>       {closed_msk.strftime('%d.%m.%Y %H:%M')} MSK"
)

# ============================================================
# POSITION BLOCK
# ============================================================

if position_direction == "NONE":
    position_block = "⚪ <b>Нет позиции</b>"

else:
    position_icon = "🟢" if position_direction == "LONG" else "🔴"

    if np.isnan(sl_price):
        position_block = (
            f"{position_icon} <b>{position_direction} × {position_qty:g}</b>\n"
            f"Вход:        {fmt_price(entry_price)}\n"
            f"⚠️ Уровни не рассчитаны"
        )
    else:
        position_block = (
            f"{position_icon} <b>{position_direction} × {position_qty:g}</b>\n"
            f"Вход:        {fmt_price(entry_price)}\n"
            f"SL:          {fmt_price(sl_price)}\n"
            f"TP:          {fmt_price(tp_price)}\n"
            f"BE:          {fmt_price(be_trigger)}"
        )

# ============================================================
# SIGNAL BLOCK
# ============================================================

if position_direction == "NONE":
    if closed["long_signal"]:
        signal_block = "🟢 <b>LONG ENTRY</b>"
    elif closed["short_signal"]:
        signal_block = "🔴 <b>SHORT ENTRY</b>"
    else:
        signal_block = "⚪ Нет сигнала"
else:
    signal_block = "📊 В позиции"

# ============================================================
# SAR BLOCK
# ============================================================

sar_trend = "LONG" if closed["sar_trend"] == 1 else "SHORT"
sar_icon = "🟢" if closed["sar_trend"] == 1 else "🔴"

sar_block = (
    f"{fmt_price(closed['sar'])} · {sar_icon} {sar_trend}"
)

# ============================================================
# ACTION BLOCK
# ============================================================

if position_direction == "NONE":
    if closed["long_signal"]:
        action = "🟢 OPEN LONG"
    elif closed["short_signal"]:
        action = "🔴 OPEN SHORT"
    else:
        action = "⚪ WAIT"
else:
    if np.isnan(entry_price):
        action = "⚠️ NO ACTION"
    else:
        if position_direction == "LONG":
            hit_sl = current_price <= sl_price
            hit_tp = current_price >= tp_price
            sar_exit = closed["sar_trend"] == -1
        else:
            hit_sl = current_price >= sl_price
            hit_tp = current_price <= tp_price
            sar_exit = closed["sar_trend"] == 1

        if hit_sl:
            action = "🔴 EXIT — SL"
        elif hit_tp:
            action = "🟢 EXIT — TP"
        elif sar_exit:
            action = "🟡 EXIT — SAR"
        else:
            action = "🟢 HOLD"

# ============================================================
# FINAL MESSAGE
# ============================================================

telegram_message = (
    f"💎 <b>GLDRUBF QUANT SENTRY</b>\n"
    f"\n"
    f"🌍 <b>Рынок</b>\n"
    f"{market_block}\n"
    f"\n"
    f"📈 <b>Позиция</b>\n"
    f"{position_block}\n"
    f"\n"
    f"🎯 <b>Сигнал</b>\n"
    f"{signal_block}\n"
    f"\n"
    f"📐 <b>SAR</b>\n"
    f"{sar_block}\n"
    f"\n"
    f"➡️ <b>Действие</b>\n"
    f"<b>{action}</b>\n"
    f"\n"
    f"⏱ {updated_msk.strftime('%H:%M:%S')} MSK"
)

# ============================================================
# SEND
# ============================================================

result = telegram_send(telegram_message)

if result.get("ok"):
    print("✅ FINAL STATUS отправлен в Telegram")
    print()
    print("Message preview:")
    print("-" * 70)
    print(telegram_message.replace("<b>", "").replace("</b>", ""))
    print("-" * 70)
else:
    print("❌ Ошибка отправки")

✅ FINAL STATUS отправлен в Telegram

Message preview:
----------------------------------------------------------------------
💎 GLDRUBF QUANT SENTRY

🌍 Рынок
💰 Цена:        11 699.00
📊 Закрытие 4H: 11 692.90
📈 EMA 100:     10 988.12
⏱ Свеча:       13.08.2026 19:00 MSK

📈 Позиция
🟢 LONG × 3
Вход:        11 659.70
SL:          11 467.99
TP:          12 043.13
BE:          11 813.07

🎯 Сигнал
📊 В позиции

📐 SAR
11 576.69 · 🟢 LONG

➡️ Действие
🟢 HOLD

⏱ 23:31:52 MSK
----------------------------------------------------------------------
